In [0]:
spark.sql("USE CATALOG workspace")

from pyspark.sql.functions import (
    col, trim, lit, when, current_timestamp,
    round, datediff
)

# Read from bronze and silver reference tables
bronze_df     = spark.table("bronze.timesheets")
silver_asn_df = spark.table("silver.assignments").select("assignment_id")

print(f"Bronze timesheets:    {bronze_df.count()}")
print(f"Silver assignments:   {silver_asn_df.count()}")
bronze_df.printSchema()

In [0]:

valid_asn_ids = [row.assignment_id for row in silver_asn_df.collect()]

MAX_HOURS = 45
MIN_HOURS = 0

cleaned_df = (
    bronze_df
    
    .withColumn("hours_valid", when(
        col("hours_logged").between(MIN_HOURS, MAX_HOURS),
        lit(True)).otherwise(lit(False)))

    #cap hours at 45 if over
    .withColumn("hours_logged_clean", when(
        col("hours_logged") > MAX_HOURS, lit(MAX_HOURS))
        .otherwise(col("hours_logged")))

  
    .withColumn("utilisation_pct", round(
        (col("hours_logged_clean") / 40) * 100, 2))

    # Validate assignment IDs exist in silver
    .withColumn("assignment_valid", when(
        col("assignment_id").isin(valid_asn_ids),
        lit(True)).otherwise(lit(False)))

    # Validate date logic
    .withColumn("date_valid", when(
        col("week_end") > col("week_start"),
        lit(True)).otherwise(lit(False)))

    # Flag non-billable entries
    .withColumn("billable_flag", when(
        col("billable") == True,
        lit("Billable")).otherwise(lit("Non-billable")))

    # Flag nulls 
    .withColumn("has_nulls", when(
        col("timesheet_id").isNull() |
        col("employee_id").isNull() |
        col("project_id").isNull() |
        col("assignment_id").isNull() |
        col("hours_logged").isNull(),
        lit(True)).otherwise(lit(False)))

    # Add audit columns
    .withColumn("silver_loaded_at", current_timestamp())
    .withColumn("source_table",     lit("bronze.timesheets"))
)

# drop duplicates for timesheet_id
cleaned_df = cleaned_df.dropDuplicates(["timesheet_id"])

print(f"Silver row count:          {cleaned_df.count()}")
print(f"Invalid hours:             {cleaned_df.filter(col('hours_valid') == False).count()}")
print(f"Invalid assignment IDs:    {cleaned_df.filter(col('assignment_valid') == False).count()}")
print(f"Invalid dates:             {cleaned_df.filter(col('date_valid') == False).count()}")
print(f"Non-billable entries:      {cleaned_df.filter(col('billable_flag') == 'Non-billable').count()}")
print(f"Rows with nulls:           {cleaned_df.filter(col('has_nulls') == True).count()}")
cleaned_df.show(5)

In [0]:
# write to silver 
(
    cleaned_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.timesheets")
)

print(f"Written to silver.timesheets: {spark.table('silver.timesheets').count()} rows")